# 04-04. Modifying the Four-box Model  
# 4-box モデルを自分で改造する

**Ocean Box Modeling with Python / 海洋ボックスモデル入門**

---

## 今日の目的 / Goals

ここまでで、4-box モデルを「使う」ところまで来ました。  
So far, we have learned how to **use** the 4-box model.

この Notebook では、次の段階へ進みます。  
In this notebook, we move to the next step.

> **モデルを自分で改造する。**  
> **Modify the model yourself.**

これは研究ではとても重要です。  
This is very important in research.

既存のモデルをそのまま使うだけではなく、  
Not only using an existing model,

```text
問いを立てる
↓
仮説を作る
↓
モデル式を少し変える
↓
実験する
↓
結果を解釈する
```

という流れを経験します。  
we experience the workflow:

```text
ask a question
↓
make a hypothesis
↓
modify the model equation slightly
↓
run experiments
↓
interpret the result
```

今日の改造は次です。  
Today we modify the model in these ways:

1. 関数をコピーして安全に改造する  
   Copy functions and modify them safely.

2. 南大洋ベンチレーションを時間変化させる  
   Make Southern Ocean ventilation time-dependent.

3. 生物ポンプに温度依存性を入れる  
   Add temperature dependence to the biological pump.

4. ガス交換速度を箱ごとに変える  
   Change gas exchange speed by box.

5. 自分のシナリオ実験を作る  
   Create your own scenario experiment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

plt.rcParams["figure.figsize"] = (7, 4)

## 1. まず最小版 4-box モデルを用意する / Prepare a compact 4-box model

この Notebook では、改造しやすいように、4-box モデルを少し整理して書きます。  
In this notebook, we write a compact version of the 4-box model so that it is easy to modify.

ここで大事なのは、**完全に現実的にすることではなく、改造しやすくすること**です。  
The important point here is **not to make the model fully realistic, but to make it easy to modify**.

研究でも、最初は小さいモデルで仮説を試すことがよくあります。  
In research, we often test hypotheses first with a small model.

In [ ]:
# ============================================================
# Compact four-box model setup
# ============================================================

CV1 = 1.0250e3
CV2 = 9.7561e-4
CV3 = 1.0e6
CV4 = 3.1536e7
CV5 = 8.64e4

def to_umolkg(x):
    return CV2 * 1.0e6 * x

def to_ppmv(x):
    return CV3 * x

# Geometry
VOCN_TOTAL = 1.292e18
AOCN = 3.49e14
VATM = 1.773e20

AOCNH = AOCN * 0.10
AOCNN = AOCN * 0.15
AOCNL = AOCN * 0.75

VOCNH = AOCNH * 250.0
VOCNN = AOCNN * 250.0
VOCNL = AOCNL * 100.0
VOCND = VOCN_TOTAL - VOCNH - VOCNN - VOCNL

VOLUME = {"H": VOCNH, "L": VOCNL, "N": VOCNN, "D": VOCND}
AREA = {"H": AOCNH, "L": AOCNL, "N": AOCNN}

TEMP = {"H": 1.0, "L": 21.5, "N": 3.0, "D": 1.75}
SALT = {"H": 34.7, "L": 34.7, "N": 34.7, "D": 34.7}

DT = 8.64e4

RCP = 106.0
RNP = 16.0
RRC_DEFAULT = 0.20
RO2P = 172.0

R = 1.0 / CV4
HSC = 2.0e-8 * CV1
DEL = 100.0

In [ ]:
# Very compact carbonate and O2 helper functions.
# These are the same style as previous notebooks.

def o2sat(TEM, SAL):
    N1, N2, N3, N4 = -1.734292e2, 2.496339e2, 1.433483e2, -2.184920e1
    N5, N6, N7 = -3.309600e-2, 1.425900e-2, -1.700000e-3
    ATEM = TEM + 273.15
    O2S = math.exp(
        N1 + N2 * 1.0e2 / ATEM
        + N3 * math.log(ATEM / 1.0e2)
        + N4 * ATEM / 1.0e2
        + SAL * (N5 + N6 * ATEM / 1.0e2 + N7 * (ATEM / 1.0e2) ** 2)
    )
    return O2S * 4.35e1 * 1.025e-3

def carbonate_simple(temp, alk, dic):
    # A deliberately simplified teaching proxy:
    # pCO2 increases with DIC, decreases with ALK, and increases with temperature.
    # 教育用の簡略 proxy: DIC で増え、ALK で減り、温度で増える。
    dic_ref = 2.235e-3 * CV1
    alk_ref = 2.374e-3 * CV1
    temp_factor = np.exp(0.0423 * (temp - 15.0))
    pco2 = 280e-6 * (dic / dic_ref) ** 2 * (alk_ref / alk) ** 1.5 * temp_factor
    co3 = 220e-6 * CV1 * (alk / alk_ref) * (dic_ref / dic)
    return {"PCO2": pco2, "CO32": co3, "K0": 0.035}

## 2. 標準モデル関数 / Standard model function

まず、標準の 4-box モデル関数を用意します。  
First, we prepare the standard 4-box model function.

ここで使う炭酸系は、計算を速くして改造しやすくするために簡略版です。  
Here we use a simplified carbonate proxy to make the notebook faster and easier to modify.

**注意 / Note**

これは教育用の簡略版です。  
This is a simplified teaching version.

前の Notebook の詳細な炭酸系関数とは異なります。  
It is different from the detailed carbonate chemistry function used in previous notebooks.

ただし、モデル改造の考え方を学ぶには十分です。  
However, it is sufficient for learning how to modify a model.

In [ ]:
def initial_four_box():
    x = {}
    for b in ["H", "L", "N", "D"]:
        x[f"PO4{b}"] = 2.10e-6 * CV1
        x[f"DIC{b}"] = 2.235e-3 * CV1
        x[f"ALK{b}"] = 2.374e-3 * CV1
        x[f"DO2{b}"] = 1.60e-4 * CV1
    x["PCO2A"] = 280.0 / CV3
    return x

def compute_exports_standard(x, CEPH=0.75, CEPN=0.02, LF=0.5):
    EPH = (CEPH / RCP) * AREA["H"] / CV4
    EPN = (CEPN / RCP) * AREA["N"] / CV4
    EPL = R * DEL * LF * x["PO4L"] * (x["PO4L"] / (HSC + x["PO4L"])) * VOLUME["L"]
    return {"H": EPH, "L": EPL, "N": EPN}

def one_step_standard(
    x,
    Tcir=2.0e7,
    FHD=6.0e7,
    LF=0.5,
    CEPH=0.75,
    CEPN=0.02,
    RRC=0.20,
    gas_scale={"H": 1.0, "L": 1.0, "N": 1.0},
    air_sea=True,
):
    y = dict(x)

    exports = compute_exports_standard(x, CEPH=CEPH, CEPN=CEPN, LF=LF)
    EPH, EPL, EPN = exports["H"], exports["L"], exports["N"]

    alk_factor = 2.0 * RRC * RCP - RNP
    dic_factor = (1.0 + RRC) * RCP

    # gas exchange
    gas = {"H": 0.0, "L": 0.0, "N": 0.0}
    if air_sea:
        for b in ["H", "L", "N"]:
            c = carbonate_simple(TEMP[b], x[f"ALK{b}"], x[f"DIC{b}"])
            piston = 3.0 * AREA[b] / CV5 * gas_scale.get(b, 1.0)
            gas[b] = piston * CV1 * c["K0"] * (x["PCO2A"] - c["PCO2"])

    def update(prefix, bio_factor, gas_terms=None):
        if gas_terms is None:
            gas_terms = {"H": 0.0, "L": 0.0, "N": 0.0}

        y[f"{prefix}H"] = x[f"{prefix}H"] + (
            Tcir * (x[f"{prefix}D"] - x[f"{prefix}H"])
            + FHD * (x[f"{prefix}D"] - x[f"{prefix}H"])
            - bio_factor * EPH
            + gas_terms.get("H", 0.0)
        ) * DT / VOLUME["H"]

        y[f"{prefix}L"] = x[f"{prefix}L"] + (
            Tcir * (x[f"{prefix}H"] - x[f"{prefix}L"])
            - bio_factor * EPL
            + gas_terms.get("L", 0.0)
        ) * DT / VOLUME["L"]

        y[f"{prefix}N"] = x[f"{prefix}N"] + (
            Tcir * (x[f"{prefix}L"] - x[f"{prefix}N"])
            - bio_factor * EPN
            + gas_terms.get("N", 0.0)
        ) * DT / VOLUME["N"]

        y[f"{prefix}D"] = x[f"{prefix}D"] + (
            Tcir * (x[f"{prefix}N"] - x[f"{prefix}D"])
            + FHD * (x[f"{prefix}H"] - x[f"{prefix}D"])
            + bio_factor * (EPH + EPL + EPN)
        ) * DT / VOLUME["D"]

    update("PO4", 1.0)
    update("ALK", alk_factor)
    update("DIC", dic_factor, gas_terms=gas)

    # O2 update
    O2SAT = {b: o2sat(TEMP[b], SALT[b]) for b in ["H", "L", "N", "D"]}
    y["DO2H"] = x["DO2H"] + (
        Tcir * (O2SAT["D"] - x["DO2H"])
        + FHD * (O2SAT["H"] - x["DO2H"])
        + RO2P * EPH
    ) * DT / VOLUME["H"]
    y["DO2L"] = x["DO2L"] + (
        Tcir * (x["DO2H"] - x["DO2L"])
        + RO2P * EPL
    ) * DT / VOLUME["L"]
    y["DO2N"] = x["DO2N"] + (
        Tcir * (x["DO2L"] - x["DO2N"])
        + RO2P * EPN
    ) * DT / VOLUME["N"]
    y["DO2D"] = x["DO2D"] + (
        Tcir * (x["DO2N"] - x["DO2D"])
        + FHD * (x["DO2H"] - x["DO2D"])
        - RO2P * (EPH + EPL + EPN)
    ) * DT / VOLUME["D"]

    if air_sea:
        flux_to_atm = 0.0
        for b in ["H", "L", "N"]:
            cnew = carbonate_simple(TEMP[b], y[f"ALK{b}"], y[f"DIC{b}"])
            piston = 3.0 * AREA[b] / CV5 * gas_scale.get(b, 1.0)
            flux_to_atm += piston * CV1 * cnew["K0"] * (cnew["PCO2"] - x["PCO2A"])
        y["PCO2A"] = x["PCO2A"] + flux_to_atm * DT / VATM

    diagnostics = {"EPH": EPH, "EPL": EPL, "EPN": EPN}
    return y, diagnostics

def diagnose(x, diag=None):
    row = {}
    for b in ["H", "L", "N", "D"]:
        c = carbonate_simple(TEMP[b], x[f"ALK{b}"], x[f"DIC{b}"])
        row[f"PO4{b}"] = to_umolkg(x[f"PO4{b}"])
        row[f"DIC{b}"] = to_umolkg(x[f"DIC{b}"])
        row[f"ALK{b}"] = to_umolkg(x[f"ALK{b}"])
        row[f"DO2{b}"] = to_umolkg(x[f"DO2{b}"])
        row[f"PCO2{b}"] = to_ppmv(c["PCO2"])
        row[f"CO3{b}"] = to_umolkg(c["CO32"])
    row["PCO2A"] = to_ppmv(x["PCO2A"])
    if diag:
        row["EPH_PgCyr"] = diag["EPH"] * RCP * 12e-15 * CV4
        row["EPL_PgCyr"] = diag["EPL"] * RCP * 12e-15 * CV4
        row["EPN_PgCyr"] = diag["EPN"] * RCP * 12e-15 * CV4
    return row

def run_model(step_func=one_step_standard, years=1500, **kwargs):
    x = initial_four_box()
    history = []

    for day in range(years * 365 + 1):
        if day % 365 == 0:
            _, diag = step_func(x, **kwargs)
            row = {"year": day / 365}
            row.update(diagnose(x, diag))
            history.append(row)

        x, diag = step_func(x, **kwargs)

    return x, pd.DataFrame(history)

## 3. 標準実験を確認する / Check the standard experiment

まずは標準モデルを走らせます。  
First, we run the standard model.

改造する前に、必ず基準となる結果を確認します。  
Before modifying a model, always check the baseline result.

In [ ]:
base_final, base = run_model(years=1500)

plt.figure()
plt.plot(base["year"], base["PCO2A"], label="Atmosphere")
plt.plot(base["year"], base["PCO2H"], label="H")
plt.plot(base["year"], base["PCO2L"], label="L")
plt.plot(base["year"], base["PCO2N"], label="N")
plt.xlabel("Time [yr]")
plt.ylabel("pCO2 [ppmv]")
plt.title("Standard 4-box model")
plt.legend()
plt.grid(True)
plt.show()

base.tail()

## 4. 改造の基本：関数をコピーする / Basic rule: copy the function first

モデルを改造するとき、元の関数を直接壊すのは危険です。  
When modifying a model, it is risky to directly edit the original function.

まずコピーを作り、そのコピーを改造します。  
First, make a copy and modify the copy.

```text
one_step_standard()
↓ copy
one_step_modified()
```

この Notebook では、改造ごとに新しい関数を作ります。  
In this notebook, we make a new function for each modification.

これにより、標準実験と改造実験を比較できます。  
This allows us to compare the standard experiment with the modified experiment.

## 5. 改造 1: 南大洋ベンチレーションを時間変化させる  
## Modification 1: Make Southern Ocean ventilation time-dependent

これまで `FHD` は一定でした。  
So far, `FHD` was constant.

しかし、氷期・退氷期のような問題では、南大洋のベンチレーションが時間変化したかもしれません。  
However, in problems such as glacial-interglacial change, Southern Ocean ventilation may have changed with time.

ここでは、`FHD` を時間とともに徐々に弱くする実験を作ります。  
Here, we create an experiment where `FHD` gradually weakens with time.

```text
FHD(t) = FHD_initial → FHD_final
```

In [ ]:
def FHD_linear_schedule(year, FHD_initial=6.0e7, FHD_final=1.5e7, transition_years=1000):
    frac = min(year / transition_years, 1.0)
    return FHD_initial + frac * (FHD_final - FHD_initial)

years = np.arange(0, 1501)
FHD_values = [FHD_linear_schedule(y) for y in years]

plt.figure()
plt.plot(years, FHD_values)
plt.xlabel("Time [yr]")
plt.ylabel("FHD [m3/s]")
plt.title("Time-dependent Southern Ocean ventilation")
plt.grid(True)
plt.show()

### 実装の考え方 / Implementation idea

通常の `run_model()` は、同じ `FHD` を毎ステップ使います。  
The usual `run_model()` uses the same `FHD` at every step.

今回は、時間ごとに `FHD` を変えたいので、専用の run 関数を作ります。  
Here, we want to change `FHD` with time, so we create a special run function.

In [ ]:
def run_time_varying_FHD(years=1500, FHD_initial=6.0e7, FHD_final=1.5e7, transition_years=1000):
    x = initial_four_box()
    history = []

    for day in range(years * 365 + 1):
        year = day / 365
        current_FHD = FHD_linear_schedule(year, FHD_initial, FHD_final, transition_years)

        if day % 365 == 0:
            _, diag = one_step_standard(x, FHD=current_FHD)
            row = {"year": year, "FHD": current_FHD}
            row.update(diagnose(x, diag))
            history.append(row)

        x, diag = one_step_standard(x, FHD=current_FHD)

    return x, pd.DataFrame(history)

final_tv, hist_tv = run_time_varying_FHD()

plt.figure()
plt.plot(base["year"], base["PCO2A"], label="standard")
plt.plot(hist_tv["year"], hist_tv["PCO2A"], label="time-varying FHD")
plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Effect of time-varying FHD")
plt.legend()
plt.grid(True)
plt.show()

### Mini exercise 1 / ミニ演習 1

`FHD_final` を `0.5e7`, `1.5e7`, `3.0e7` に変えてみてください。  
Try changing `FHD_final` to `0.5e7`, `1.5e7`, and `3.0e7`.

大気 pCO2 の低下量はどう変わるでしょうか。  
How does the decrease in atmospheric pCO2 change?

In [ ]:
FHD_final_list = [0.5e7, 1.5e7, 3.0e7]
summary = []

plt.figure()
for fhd_final in FHD_final_list:
    _, h = run_time_varying_FHD(FHD_final=fhd_final)
    plt.plot(h["year"], h["PCO2A"], label=f"FHD_final={fhd_final:.1e}")
    summary.append({
        "FHD_final": fhd_final,
        "Final pCO2A": h["PCO2A"].iloc[-1],
        "Final DO2D": h["DO2D"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Sensitivity to final FHD")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary)

## 6. 改造 2: 生物ポンプに温度依存性を入れる  
## Modification 2: Add temperature dependence to the biological pump

これまで、輸出生産は主に PO4 やパラメタで決まっていました。  
So far, export production was controlled mainly by PO4 and parameters.

しかし、生物過程は温度に依存する可能性があります。  
However, biological processes may depend on temperature.

ここでは簡単に、温度が高いほど輸出生産が強くなる形を入れます。  
Here we add a simple form in which export becomes stronger at higher temperature.

\[
f_T = Q_{10}^{(T - T_{\mathrm{ref}})/10}
\]

ここでは \(Q_{10}=2\) とします。  
Here we use \(Q_{10}=2\).

In [ ]:
def temp_factor(temp, tref=15.0, q10=2.0):
    return q10 ** ((temp - tref) / 10.0)

temps = np.linspace(0, 25, 100)
factors = [temp_factor(t) for t in temps]

plt.figure()
plt.plot(temps, factors)
plt.xlabel("Temperature [deg C]")
plt.ylabel("Temperature factor")
plt.title("Q10 temperature dependence")
plt.grid(True)
plt.show()

In [ ]:
def compute_exports_temperature_dependent(x, CEPH=0.75, CEPN=0.02, LF=0.5, q10=2.0):
    base = compute_exports_standard(x, CEPH=CEPH, CEPN=CEPN, LF=LF)

    # Apply temperature dependence only to surface boxes
    # 表層ボックスの輸出生産に温度依存性をかける
    out = {}
    for b in ["H", "L", "N"]:
        out[b] = base[b] * temp_factor(TEMP[b], tref=15.0, q10=q10)
    return out

pd.DataFrame({
    "Box": ["H", "L", "N"],
    "Temperature": [TEMP[b] for b in ["H", "L", "N"]],
    "Standard export [PgC/yr]": [
        compute_exports_standard(initial_four_box())[b] * RCP * 12e-15 * CV4 for b in ["H", "L", "N"]
    ],
    "Temperature-dependent export [PgC/yr]": [
        compute_exports_temperature_dependent(initial_four_box())[b] * RCP * 12e-15 * CV4 for b in ["H", "L", "N"]
    ],
})

### 実装 / Implementation

`one_step_standard()` の中では `compute_exports_standard()` を使っていました。  
Inside `one_step_standard()`, we used `compute_exports_standard()`.

ここでは、輸出生産だけを温度依存版に差し替えた関数を作ります。  
Here, we create a function where only export production is replaced by the temperature-dependent version.

In [ ]:
def one_step_temp_biology(x, q10=2.0, **kwargs):
    # Start from parameters
    Tcir = kwargs.get("Tcir", 2.0e7)
    FHD = kwargs.get("FHD", 6.0e7)
    LF = kwargs.get("LF", 0.5)
    CEPH = kwargs.get("CEPH", 0.75)
    CEPN = kwargs.get("CEPN", 0.02)
    RRC = kwargs.get("RRC", 0.20)

    y = dict(x)

    exports = compute_exports_temperature_dependent(x, CEPH=CEPH, CEPN=CEPN, LF=LF, q10=q10)
    EPH, EPL, EPN = exports["H"], exports["L"], exports["N"]

    alk_factor = 2.0 * RRC * RCP - RNP
    dic_factor = (1.0 + RRC) * RCP

    def update(prefix, bio_factor):
        y[f"{prefix}H"] = x[f"{prefix}H"] + (
            Tcir * (x[f"{prefix}D"] - x[f"{prefix}H"])
            + FHD * (x[f"{prefix}D"] - x[f"{prefix}H"])
            - bio_factor * EPH
        ) * DT / VOLUME["H"]
        y[f"{prefix}L"] = x[f"{prefix}L"] + (
            Tcir * (x[f"{prefix}H"] - x[f"{prefix}L"])
            - bio_factor * EPL
        ) * DT / VOLUME["L"]
        y[f"{prefix}N"] = x[f"{prefix}N"] + (
            Tcir * (x[f"{prefix}L"] - x[f"{prefix}N"])
            - bio_factor * EPN
        ) * DT / VOLUME["N"]
        y[f"{prefix}D"] = x[f"{prefix}D"] + (
            Tcir * (x[f"{prefix}N"] - x[f"{prefix}D"])
            + FHD * (x[f"{prefix}H"] - x[f"{prefix}D"])
            + bio_factor * (EPH + EPL + EPN)
        ) * DT / VOLUME["D"]

    update("PO4", 1.0)
    update("ALK", alk_factor)
    update("DIC", dic_factor)

    # Keep O2 simple here
    y["DO2H"] = x["DO2H"] + (RO2P * EPH) * DT / VOLUME["H"]
    y["DO2L"] = x["DO2L"] + (RO2P * EPL) * DT / VOLUME["L"]
    y["DO2N"] = x["DO2N"] + (RO2P * EPN) * DT / VOLUME["N"]
    y["DO2D"] = x["DO2D"] - (RO2P * (EPH + EPL + EPN)) * DT / VOLUME["D"]

    # Atmosphere is diagnosed with simplified gas exchange by calling standard gas update approximately disabled here
    cH = carbonate_simple(TEMP["H"], y["ALKH"], y["DICH"])
    cL = carbonate_simple(TEMP["L"], y["ALKL"], y["DICL"])
    cN = carbonate_simple(TEMP["N"], y["ALKN"], y["DICN"])
    flux_to_atm = 0.0
    for b, c in zip(["H", "L", "N"], [cH, cL, cN]):
        piston = 3.0 * AREA[b] / CV5
        flux_to_atm += piston * CV1 * c["K0"] * (c["PCO2"] - x["PCO2A"])
    y["PCO2A"] = x["PCO2A"] + flux_to_atm * DT / VATM

    return y, {"EPH": EPH, "EPL": EPL, "EPN": EPN}

final_q10, hist_q10 = run_model(step_func=one_step_temp_biology, years=1500, q10=2.0)

plt.figure()
plt.plot(base["year"], base["PCO2A"], label="standard")
plt.plot(hist_q10["year"], hist_q10["PCO2A"], label="temperature-dependent biology")
plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Adding temperature dependence to export")
plt.legend()
plt.grid(True)
plt.show()

### Mini exercise 2 / ミニ演習 2

`q10` を 1.0, 2.0, 3.0 に変えてください。  
Change `q10` to 1.0, 2.0, and 3.0.

低緯度 L の輸出生産と大気 pCO2 はどう変わるでしょうか。  
How do low-latitude export and atmospheric pCO2 change?

In [ ]:
q10_list = [1.0, 2.0, 3.0]
summary = []

plt.figure()
for q10 in q10_list:
    _, h = run_model(step_func=one_step_temp_biology, years=1500, q10=q10)
    plt.plot(h["year"], h["PCO2A"], label=f"q10={q10}")
    summary.append({
        "q10": q10,
        "Final pCO2A": h["PCO2A"].iloc[-1],
        "Final EPL": h["EPL_PgCyr"].iloc[-1],
        "Final DICD": h["DICD"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Sensitivity to Q10")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary)

## 7. 改造 3: ガス交換を箱ごとに変える  
## Modification 3: Change gas exchange by box

南大洋、低緯度、北大西洋では風や海氷の状態が違います。  
Southern Ocean, low latitudes, and North Atlantic have different winds and sea-ice conditions.

そのため、ガス交換速度も地域によって違うはずです。  
Therefore, gas exchange speed should differ by region.

ここでは `gas_scale` を使って、箱ごとのガス交換速度を変えます。  
Here we use `gas_scale` to change gas exchange speed by box.

```python
gas_scale = {"H": 0.3, "L": 1.0, "N": 1.0}
```

これは、南大洋 H のガス交換を 30% に弱める実験です。  
This weakens gas exchange in H to 30%.

In [ ]:
gas_cases = {
    "standard": {"H": 1.0, "L": 1.0, "N": 1.0},
    "weak H gas": {"H": 0.3, "L": 1.0, "N": 1.0},
    "strong H gas": {"H": 2.0, "L": 1.0, "N": 1.0},
    "weak all gas": {"H": 0.3, "L": 0.3, "N": 0.3},
}

summary = []
plt.figure()

for name, scale in gas_cases.items():
    _, h = run_model(years=1500, gas_scale=scale)
    plt.plot(h["year"], h["PCO2A"], label=name)
    summary.append({
        "Case": name,
        "Final pCO2A": h["PCO2A"].iloc[-1],
        "Final PCO2H": h["PCO2H"].iloc[-1],
        "Final PCO2L": h["PCO2L"].iloc[-1],
        "Final PCO2N": h["PCO2N"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Changing gas exchange by box")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary)

### Mini exercise 3 / ミニ演習 3

南大洋 H のガス交換を弱めると、大気 pCO2 はどう変わりましたか。  
What happened to atmospheric pCO2 when gas exchange in H was weakened?

この結果は、H が深層水と大気の接点であることと関係していますか。  
Is this related to the fact that H is a contact point between deep water and the atmosphere?

## 8. 改造 4: 複合シナリオを作る  
## Modification 4: Create a combined scenario

実際の研究では、1 つのパラメタだけが変わるとは限りません。  
In real research, not only one parameter changes.

例えば氷期には、次のような変化が同時に起きたかもしれません。  
For example, during glacial periods, several changes may have occurred simultaneously.

```text
Southern Ocean ventilation weakens
Southern Ocean gas exchange weakens
biological export strengthens
CaCO3 rain ratio changes
```

ここでは、それらを組み合わせたシナリオを作ります。  
Here we create a combined scenario.

In [ ]:
scenario_cases = {
    "baseline": {},
    "weak ventilation": {"FHD": 1.5e7},
    "weak H gas": {"gas_scale": {"H": 0.3, "L": 1.0, "N": 1.0}},
    "strong biology": {"LF": 1.0, "CEPH": 1.5},
    "combined": {
        "FHD": 1.5e7,
        "LF": 1.0,
        "CEPH": 1.5,
        "gas_scale": {"H": 0.3, "L": 1.0, "N": 1.0},
    },
}

summary = []
plt.figure()

for name, kwargs in scenario_cases.items():
    _, h = run_model(years=1500, **kwargs)
    plt.plot(h["year"], h["PCO2A"], label=name)
    summary.append({
        "Case": name,
        "Final pCO2A": h["PCO2A"].iloc[-1],
        "Final DICD": h["DICD"].iloc[-1],
        "Final DO2D": h["DO2D"].iloc[-1],
        "Final PO4H": h["PO4H"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Combined scenario experiments")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary)

### 考察 / Interpretation

複合シナリオでは、複数の過程が同時に変わります。  
In a combined scenario, multiple processes change at the same time.

そのため、結果の解釈は難しくなります。  
Therefore, interpretation becomes harder.

しかし、現実の気候変化は複合的です。  
However, real climate change is also combined and complex.

**研究者として重要なこと / Important as a researcher**

複合実験をする前に、必ず単独実験を行います。  
Before running a combined experiment, always run single-factor experiments.

```text
single-factor experiments
↓
combined experiment
↓
interpretation
```

この順番が大事です。  
This order is important.

## 9. 自分の仮説を作る / Make your own hypothesis

ここからは自由研究です。  
From here, this is your own research exercise.

まず、仮説を 1 つ書いてください。  
First, write one hypothesis.

例:  
Example:

> 南大洋のベンチレーションが弱くなり、同時に生物ポンプが強くなると、大気 CO2 は大きく低下する。  
> If Southern Ocean ventilation weakens and the biological pump strengthens at the same time, atmospheric CO2 will strongly decrease.

次に、その仮説をパラメタに翻訳します。  
Next, translate the hypothesis into parameters.

```text
FHD ↓
CEPH ↑
LF ↑
gas_scale["H"] ↓
```

In [ ]:
# Your own scenario
# 自分のシナリオ

my_scenario = {
    "FHD": 1.5e7,
    "LF": 1.0,
    "CEPH": 1.5,
    "RRC": 0.2,
    "gas_scale": {"H": 0.3, "L": 1.0, "N": 1.0},
}

final_my, hist_my = run_model(years=1500, **my_scenario)

plt.figure()
plt.plot(base["year"], base["PCO2A"], label="baseline")
plt.plot(hist_my["year"], hist_my["PCO2A"], label="my scenario")
plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("My modified 4-box model scenario")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame({
    "Quantity": ["pCO2A", "DICD", "DO2D", "PO4H", "PO4L"],
    "Baseline": [
        base["PCO2A"].iloc[-1],
        base["DICD"].iloc[-1],
        base["DO2D"].iloc[-1],
        base["PO4H"].iloc[-1],
        base["PO4L"].iloc[-1],
    ],
    "My scenario": [
        hist_my["PCO2A"].iloc[-1],
        hist_my["DICD"].iloc[-1],
        hist_my["DO2D"].iloc[-1],
        hist_my["PO4H"].iloc[-1],
        hist_my["PO4L"].iloc[-1],
    ],
})

## 10. 改造したモデルを短く報告する / Write a short report

モデル改造をしたら、結果を短く報告します。  
After modifying a model, write a short report.

研究報告では、次の順番が分かりやすいです。  
A clear research report follows this order:

```text
1. 問い
2. 仮説
3. 改造した点
4. 実験設定
5. 結果
6. 解釈
7. 限界
```

```text
1. Question
2. Hypothesis
3. Model modification
4. Experimental setup
5. Results
6. Interpretation
7. Limitations
```

### テンプレート / Template

```text
問い:
  ...

仮説:
  ...

改造した点:
  ...

実験設定:
  ...

結果:
  ...

解釈:
  ...

限界:
  ...
```

## 11. 課題 / Exercises

### 課題 1 / Exercise 1

なぜモデルを改造するとき、元の関数を直接書き換えず、コピーを作るべきなのか。  
Why should we copy a function instead of directly modifying the original function?

### 課題 2 / Exercise 2

時間変化する `FHD` を入れることで、どのような科学的問題を表現できるか。  
What scientific problem can be represented by introducing time-varying `FHD`?

### 課題 3 / Exercise 3

生物ポンプに温度依存性を入れると、どの box の輸出生産が特に変わりやすいか。  
When temperature dependence is added to the biological pump, which box's export production changes most easily?

### 課題 4 / Exercise 4

南大洋 H のガス交換を弱める実験は、どのような現実過程の簡略表現と考えられるか。  
Weakening gas exchange in H can be interpreted as a simplified representation of what real-world process?

### 課題 5 / Exercise 5

自分のシナリオ実験について、問い・仮説・結果・解釈を 5〜10 行でまとめよ。  
Summarize your own scenario experiment in 5–10 lines: question, hypothesis, result, and interpretation.

---

## 想定解答 / Expected answers

### 課題 1

元の関数を直接書き換えると、標準実験を再現できなくなる。コピーを作れば、標準実験と改造実験を比較でき、バグが出たときにも戻りやすい。  
If the original function is directly modified, the standard experiment may no longer be reproducible. By making a copy, we can compare the standard and modified experiments and recover more easily from bugs.

### 課題 2

退氷期や氷期のように、南大洋の深層ベンチレーションが時間とともに変化する問題を表現できる。  
It can represent problems such as glacial or deglacial changes in which Southern Ocean ventilation changes through time.

### 課題 3

温度依存性を入れると、暖かい低緯度 L box の輸出生産が特に強く変化しやすい。  
With temperature dependence, export production in the warm low-latitude L box tends to change strongly.

### 課題 4

南大洋 H のガス交換を弱めることは、海氷の拡大、風速低下、成層化による大気海洋交換の抑制などの簡略表現と考えられる。  
Weakening gas exchange in H can be viewed as a simplified representation of sea-ice expansion, weaker winds, or stronger stratification suppressing air-sea exchange.

### 課題 5

解答例: 南大洋ベンチレーションを弱め、生物ポンプを強めると、大気 CO2 は低下した。これは、深層に蓄積した炭素が大気へ出にくくなり、さらに表層炭素が深層へ輸送されたためと解釈できる。ただし、このモデルは空間構造や海氷、鉄制限を明示的に含まないため、現実の氷期 CO2 変化を完全に説明するものではない。  
Example answer: When Southern Ocean ventilation was weakened and the biological pump was strengthened, atmospheric CO2 decreased. This can be interpreted as deep carbon becoming less able to reach the atmosphere, while surface carbon was additionally exported to the deep ocean. However, this model does not explicitly include spatial structure, sea ice, or iron limitation, so it cannot fully explain real glacial CO2 change.

## 12. 次へ / Next step

この Notebook では、4-box モデルを自分で改造しました。  
In this notebook, we modified the 4-box model ourselves.

次は、4-box シリーズの総まとめとして、短い研究発表に近い形へ進みます。  
Next, we move toward a short research-presentation style summary of the 4-box series.

```text
04-05:
  research-style synthesis
  compare scenarios
  prepare figures
  write a short interpretation
```

ここまで来ると、学生は単にコードを実行するだけでなく、  
At this point, students are no longer just running code,

> **モデルを使って問いを立てる**

段階に入ります。  
but are entering the stage of:

> **asking questions using a model**